# Kendi AI Modelini Eğit - Colab Notebook

Bu notebook, proje dosyalarını Google Drive'dan alarak CPT ve SFT eğitimini çalıştırır.

**Kullanım:**
1. `colab_ready` klasörünü Google Drive kök dizinine `ai-model-colab` olarak yükleyin.
2. Veri ve tokenizer dosyalarınızı hazırlayıp aynı klasöre koyun.
3. Bu notebooku Drive'dan açın.
4. GPU çalışma zamanı seçin (T4 veya L4).
5. `Çalışma zamanı → Tümünü çalıştır` veya hücreleri sırayla çalıştırın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Ayarlar ve Veri Kontrolü

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/ai-model-colab'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

RUN_CPT = True
RUN_SFT = True

required = {
    'Tokenizer': Path(PROJECT_DIR) / 'models' / 'my-tokenizer' / 'tokenizer.model',
    'CPT corpus': Path(PROJECT_DIR) / 'data' / 'corpus' / 'my_corpus.pt',
    'CPT meta': Path(PROJECT_DIR) / 'data' / 'corpus' / 'my_corpus_meta.json',
    'SFT data': Path(PROJECT_DIR) / 'data' / 'sft' / 'sft_data.jsonl',
}

missing = [name for name, p in required.items() if not p.exists()]
if missing:
    print('
[ERROR] The following required files are missing:')
    for name in missing:
        print(f'  - {name}')
    print('
Please prepare your data and tokenizer first.')
    print('See colab_ready/README.md for instructions.')
    raise SystemExit
else:
    print('All required files found. Ready to train.')

best_cpt = Path(PROJECT_DIR) / 'checkpoints' / 'best_cpt'
final_model = Path(PROJECT_DIR) / 'models' / 'my-model-final'
print('best_cpt exists:', best_cpt.exists())
print('final_model exists:', final_model.exists())

## Gereksinimler

In [ ]:
!pip install -q -r requirements.txt

## CPT (Continued Pre-Training)

In [ ]:
import subprocess

if not RUN_CPT:
    print('[CPT] Skipped by user setting.')
elif best_cpt.exists():
    print('[CPT] best_cpt already exists, skipping CPT.')
else:
    print('[CPT] Starting training...')
    subprocess.run(['python', 'scripts/train_cpt.py', '--config', 'configs/training.yaml'], check=True)

## SFT (Supervised Fine-Tuning)

In [ ]:
if not RUN_SFT:
    print('[SFT] Skipped by user setting.')
elif final_model.exists():
    print('[SFT] Final model already exists, skipping SFT.')
else:
    print('[SFT] Starting training...')
    subprocess.run(['python', 'scripts/train_sft.py', '--config', 'configs/training.yaml'], check=True)

## Hızlı Test

In [ ]:
from transformers import AutoTokenizer, LlamaForCausalLM
import torch

model_dir = 'models/my-model-final'
if not Path(model_dir).exists():
    print(f'[TEST] {model_dir} not found. Train SFT first.')
else:
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = LlamaForCausalLM.from_pretrained(model_dir).to('cuda' if torch.cuda.is_available() else 'cpu')
    messages = [{'role': 'user', 'content': "Türkiye'nin başkenti neresidir?"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))